# Rivet Notebook
**Bayes_HEP: Rivet_Main**

<details>
<summary>
What is this notebook?
</summary>

This notebook runs a notebook version of Rivet_Main, which is the **first half** of the Bayesian tuning pipeline. It uses **Pythia8** (a Monte Carlo event generator that simulates particle collisions) and **Rivet** (a particle physics analysis framework) to produce the simulated data needed for calibration.

> **New to this?** A *Monte Carlo event generator* like Pythia8 uses random numbers to simulate what happens when two particles collide — it models the physics of quark-gluon interactions, hadronization, and multi-parton interactions. Rivet then applies the same analysis cuts and observable definitions as the real experiment, so the simulation output can be directly compared to real data.

</details>

<details>
<summary>
Why do we need this?
</summary>

To tune Pythia8's free parameters, we need to know what it *predicts* across many different parameter settings. Running Pythia8 for every possible combination would take too long, so instead we run it at a small set of carefully chosen **design points** — a statistical sample of the parameter space. The outputs from these runs are used in `Bayes_Notebook.ipynb` to train a fast surrogate model (the **emulator**).

</details>

<details>
<summary>
What does this notebook do?
</summary>

1. **Configuration** — set your project paths and choose which stages to run
2. **Design point generation** — sample the Pythia8 parameter space using Latin Hypercube Sampling (LHS)
3. **Rivet analyses setup** — compile the custom C++ code that measures physics observables from Pythia8 events
4. **Run Pythia8 + Rivet** — simulate particle collisions at each design point and record the observables
5. **Merge & report** — combine output `.yoda` files and generate HTML diagnostic plots
6. **Write input files** — extract the Data and Prediction files that the Bayes notebook reads

</details>

## Workflow

Run this notebook **before** `Bayes_Notebook.ipynb`. The `input/Data/` and `input/Prediction/` files produced here feed directly into the Bayesian calibration pipeline.

---

## Companion Slides

<details>
<summary>
The `Bayes_HEP_Project.pptx` slide deck provides background for each section of this notebook. Refer to these slides as you work through the cells:
</summary>

| Notebook section | Relevant slides |
|-----------------|----------------|
| Full workflow overview | **Slides 3–6** — Bayesian inference diagram, Rivet_Main.py steps |
| Setup & tools | **Slides 11–15** — VS Code, Docker, Apptainer, ISAAC HPC, remote access |
| Physics background | **Slides 16–20** — QCD, QGP, RHIC/LHC, recommended papers |
| Rivet analyses | **Slides 21–23** — What is Rivet?, analysis file types (.cc, .yoda, .plot, .info) |
| Design points | **Slides 26–27** — Latin Hypercube Sampling, DETMAX algorithm |
| Run Pythia8 | **Slides 25, 28–30** — Monte Carlo method, Pythia8, JETSCAPE, Herwig |
| Data & observables | **Slides 31–35** — sPHENIX, jets/RAA, experimental data, HEPData |

</details>

### Step 1 — Setup (run once)

<details>
<summary>From the <code>Bayes_HEP</code> directory on ISAAC, run:</summary>

```bash
bash New_Project/drivers/notebook/utilities/setup.sh <username>
```

Replace `<username>` with your ISAAC username (e.g. `cbaillar`).

</details>

### Step 2 — Start Jupyter

<details>
<summary>In your ISAAC terminal, run:</summary>

```bash
jupyter notebook --no-browser --port=8888 --ip=0.0.0.0
```

Copy the full URL printed:
```
http://127.0.0.1:8888/?token=abc123...
```

</details>

### Step 3 — Open the notebook in VSCode

<details>
<summary>Connect to Jupyter server</summary>

1. Open the notebook via Remote-SSH in VSCode
2. Click the kernel selector in the top right corner
3. Select **Existing Jupyter Server**
4. Paste the token URL from Step 2
5. Select kernel
- For physics/analysis cells → select **Bayes HEP (Apptainer)**
- For SLURM job submission cells → select **Python 3**

> **Note:** The Jupyter server must be running before opening the notebook in VSCode.

</details>

## 1. Configuration

**This is the main cell you need to edit for your own project.** Set your paths and choose which pipeline stages to run.

### Paths

<details>
<summary>
Path details:
</summary>

#
| Variable | What to set |
|----------|-------------|
| `username` | Your ISAAC HPC username (the part after `/UTK0244/`) |
| `work_dir` | Base directory on ISAAC where your Bayes_HEP project lives |
| `main_dir` | The project directory (auto-built from `work_dir`) |
| `input` | Name of the input subdirectory inside the project directory |

</details>

### Pipeline Toggles
<details>
<summary>
Each boolean turns a pipeline stage on (<code>True</code>) or off (<code>False</code>). For a clean first run, set all to <code>True</code> and run top to bottom. On subsequent runs you can skip stages you've already completed.
</summary>

#

| Flag | What it does |
|------|--------------|
| `clear_rivet_models` | Wipes `rivet/Models/` before running — ensures no stale outputs from a previous run. Set `False` to preserve existing model runs. |
| `Rivet_Setup` | Compile custom Rivet analysis C++ code. Only needed once, or after modifying `.cc` files. |
| `Get_Design_Points` | `True` → generate new design points via LHS. `False` → load the most recent existing design file. |
| `Submit_Job_Model` | Submit SLURM batch jobs to run Pythia8 + Rivet across all design points. Recommended for full production runs. |
| `Rerun_job_model` | Resubmit SLURM jobs for specific design points listed in `DP_LIST`. Use to recover failed or incomplete runs without rerunning everything. |
| `Run_Model` | Run Pythia8 + Rivet interactively for each design point. Use for small tests only — not suitable for large design point sets. |
| `Rivet_Merge` | Merge `.yoda` files per design point and generate HTML diagnostic reports. |
| `equiv_on` | Passes the `-e` flag to `rivet-merge`, enabling equivalent weight merging. Use for minimum bias analyses. |
| `Write_input_Rivet` | Parse HTML reports and write `Data/` and `Prediction/` input files for the GP emulator. |

</details>

### Run Parameters

<details>
<summary>
Key parameters that control Pythia8 event generation and HPC job submission (set in Section 6):
</summary>

#

| Variable | Description |
|----------|-------------|
| `nevents` | Number of simulated collision events per design point. More events = less statistical noise, but longer runtime. Use 1,000 for testing; 100,000+ for production. |
| `model_seed` | Random seed for the Pythia8 event generator. Fixing it makes runs reproducible across reruns. |
| `QOS` | SLURM Quality of Service tier: `'short'` (3 h), `'campus'` (24 h), `'long'` (144 h). Choose based on expected runtime. |
| `RUN_PT_HAT_BINS` | `True` → split events into $\hat{p}_T$ bins for hard-process analyses. `False` → run inclusive (no $\hat{p}_T$ cut). |

</details>

### Required Input Files

Before running, configure these three files in `input/Rivet/`:

<details>
<summary><b>📄 analyses_list.txt</b> — which Rivet analyses and observables to run</summary>

Maps each collision system tag to analysis names and histogram IDs to extract. Format:

```
pp_200:
STAR_2020_I1783875   d01-x01-y01 d02-x01-y01
STAR_2021_I1853218   d01-x01-y01
```

**Edit:**
- Add/change analysis names (must have a matching `.cc` file in the Rivet analyses directory)
- Add histogram IDs for each analysis (e.g., `d01-x01-y01`)
- Add new system tags (e.g., `ppbar_1960:`) for different collision energies

Analysis names follow `EXPERIMENT_YEAR_InspireID`. Find them on [HEPData](https://www.hepdata.net) or [Inspire](https://inspirehep.net).

</details>

<details>
<summary><b>📄 parameter_prior_list.dat</b> — parameter names and prior ranges</summary>

Defines the 5 Pythia8 parameters being tuned and the range sampled during design point generation:

```
# Parameter pT0Ref ecmPow coreRadius coreFraction CRrange
# - Parameter pT0Ref:       Linear [0.5, 2.5]
# - Parameter ecmPow:       Linear [0.0, 0.25]
# - Parameter coreRadius:   Linear [0.1, 1.0]
# - Parameter coreFraction: Linear [0.0, 1.0]
# - Parameter CRrange:      Linear [1.0, 9.0]
```

**Edit:**
- Adjust `[min, max]` bounds to widen or narrow the search region
- Add/remove parameters if tuning a different set of Pythia8 settings
- Parameter names here must match the placeholders in `parameter.cmnd`

`Linear` prior means LHS samples uniformly from the given range.

</details>

<details>
<summary><b>📄 parameter.cmnd</b> — Pythia8 beam and physics settings</summary>

Sets beam configuration and fixed physics switches. Tuning parameter values are placeholders — overwritten automatically per design point at runtime:

```
Beams:idA = 2212                        # beam particle (2212 = proton)
Beams:idB = -2212                       # target particle (-2212 = antiproton)
MultipartonInteractions:ecmRef = 200    # collision energy in GeV
SoftQCD:all = on
MultipartonInteractions:pT0Ref = 1      # placeholder — DO NOT change
MultipartonInteractions:ecmPow = 1      # placeholder — DO NOT change
```

**Edit:**
- `Beams:idA` / `Beams:idB` — PDG particle IDs: `2212` proton, `-2212` antiproton, `1000822080` Pb-208
- `ecmRef` — center-of-mass energy in GeV (must match the system tag, e.g., `200` for `pp_200`)
- Leave all tuning parameter values as `1` — they are replaced automatically

</details>

In [ ]:
# Run for both kernels

username    = 'cbaillar'
project     = 'New_Project'
input       = 'input'

work_dir    = f'/lustre/isaac24/proj/UTK0244/{username}/Bayes_HEP'
main_dir    = f'{work_dir}/{project}'
input_dir   = f'{main_dir}/{input}'

hpc_dir=f"{main_dir}/Batch_Jobs/HPC/notebook"
CONTAINER=f"{work_dir}/bayes_hep.sif"
BIND_PATH=f"{work_dir}:/workdir"

QOS         = 'campus'

QOS_WALLTIMES = {
    'short'  : '2:55:00',
    'campus' : '23:55:00',
    'long'   : '143:55:00'
}

WALLTIME = QOS_WALLTIMES[QOS]

error_path  = f"/lustre/isaac24/scratch/{username}/jobs/error/job.e%A_%a"
output_path = f"/lustre/isaac24/scratch/{username}/jobs/output/job.o%A_%a"

In [ ]:
# Run for both kernels

clear_rivet_models  = False   # wipe rivet/Models directory before running

model               = 'pythia8'
Coll_System         = ['pp_200']      # e.g. ['pp_200', 'pp_7000']

Rivet_Setup         = True   # (re)build Rivet analyses

######## Design Points
Get_Design_Points   = True   # True → LHS; False → load from file

num_DP              = 5     # number of LHS design points
seed                = 43     # LHS seed

######## Rivet + Pythia
RUN_PT_HAT_BINS     = False
nevents             = 1000

Submit_Job_Model    = False    # Submits Slurm script for batch jobs to run Pythia8 + Rivet
Rerun_job_model     = False   # Submits Slurm script for rerunning batch jobs 
Run_Model           = True   # run Pythia8 + Rivet for each design point interactively

######## Rivet
Submit_Merge_job    = False
Rivet_Merge         = True   # merge .yoda files and generate HTML reports
equiv_on            = False
Write_input_Rivet   = True

In [ ]:
# Run for both kernels

import os
import shutil
import subprocess
import sys
import glob
import random
import numpy as np

def get_kernel():
    return 'apptainer' if 'apptainer' in sys.executable.lower() or \
           os.path.exists('/usr/local/share/Bayes_HEP') else 'host'


In [ ]:
# Run in apptainer kernel only

if get_kernel() == 'apptainer':
    from Bayes_HEP.Design_Points import reader as Reader
    from Bayes_HEP.Design_Points import design_points as DesignPoints
    from Bayes_HEP.Design_Points import plots as Plots
    from Bayes_HEP.Design_Points import rivet_html_parser as RivetParser

else:
    print("⚠️ Switch to Bayes HEP (Apptainer) kernel to run physics cells.")


## 2. Directory Setup

Creates the `rivet/` working directory where all Pythia8 and Rivet outputs are stored.

If `clear_rivet_models = True`, the `rivet/Models/` folder is deleted and recreated from scratch. This is recommended when starting a new run to avoid mixing outputs from different parameter sets. Set it to `False` if you only want to re-run later stages (like merging or writing input files) without repeating the model runs.

In [ ]:
# Run for both kernels

os.makedirs(f"{main_dir}/rivet", exist_ok=True)

models_dir = f"{main_dir}/rivet/Models"
if clear_rivet_models and os.path.exists(models_dir):
    print(f"Clearing: {models_dir}")
    shutil.rmtree(models_dir)

## 3. Parse Analyses List

Reads `input/Rivet/analyses_list.txt`, which specifies which Rivet analyses to run for each collision system.

<details>
<summary>
More information
</summary>

A **Rivet analysis** is a piece of C++ code that processes simulated particle collision events and computes physics observables — quantities like particle multiplicity distributions, transverse momentum spectra, or pseudorapidity distributions. Each analysis corresponds to a specific published experimental measurement (e.g., `STAR_2019_I1771348` is a STAR experiment paper from 2019). The analysis code applies the same kinematic cuts and binning as the real experiment so that simulation and data are directly comparable.

The `analyses_list.txt` file organizes analyses by collision system (e.g., `pp_200:` for proton-proton collisions at 200 GeV), followed by each analysis name and the specific histogram identifiers to extract. Only systems listed in `Coll_System` are processed.

#

> **Background:** See **slides 21–23** for an explanation of what Rivet is and what each analysis file type (.cc, .yoda, .plot, .info) contains.

</details>

In [ ]:
# Run for both kernels

Rivet_dir     = f'{input_dir}/Rivet'
project_dir   = f'{main_dir}/rivet'
analyses_file = 'analyses_list.txt'

os.makedirs(project_dir, exist_ok=True)

tagged_analyses = {}
analyses_list   = {}
system_tag      = None

with open(f"{Rivet_dir}/{analyses_file}", 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if line.endswith(':'):
            system_tag = line[:-1]
            tagged_analyses[system_tag] = {}
            if system_tag in Coll_System:
                analyses_list[system_tag] = []
        elif system_tag is not None:
            parts    = line.split()
            analysis = parts[0]
            histograms = parts[1:]
            tagged_analyses[system_tag][analysis] = histograms
            if system_tag in Coll_System:
                analyses_list[system_tag].append(analysis)
        else:
            raise ValueError(f"Analysis line found before system tag: {line}")

missing = [s for s in Coll_System if s not in analyses_list]
if missing:
    raise ValueError(f"Missing analyses for system(s): {missing}")

for sys_key, anals in analyses_list.items():
    print(f"{sys_key}: {anals}")

## 4. Rivet Setup — Build Analyses

Compiles the custom Rivet analyses from C++ source files (`.cc`) into shared libraries (`.so`) that Rivet loads at runtime.

<details>
<summary>
More information
</summary>

You need to re-run this step if:
- This is your first time running the notebook
- You have modified or added a `.cc` analysis file
- The Rivet version or compiler environment has changed

If a build fails, an error is raised here. The full compiler output is in `rivet/analyses.log` — look for lines ending in `build_failed` to see which analysis had a problem and what the error was.

</details>

In [ ]:
# Run in apptainer kernel only

if Rivet_Setup:
    all_analyses = [a for sys_key in analyses_list for a in analyses_list[sys_key]]
    print(f"Building analyses: {all_analyses}")

    subprocess.run(['bash', '/usr/local/share/Bayes_HEP/Design_Points/Rivet_Analyses/run_analysis.sh', ','.join(all_analyses),
        project_dir], check=True)

# Always parse the build log
with open(f"{project_dir}/analyses.log", 'r') as f:
    analyses_results = f.read().splitlines()

successful_builds = [l.split()[0] for l in analyses_results if l.strip().endswith('build_success')]
failed_builds     = [l.split()[0] for l in analyses_results if l.strip().endswith('build_failed')]

print(f"Successful builds : {successful_builds}")
if failed_builds:
    raise RuntimeError(f"Failed builds: {failed_builds}")
else:
    print("No failed builds.")

## 5. Design Points

**Design points** are the specific parameter combinations at which we run Pythia8. Instead of sampling the parameter space randomly, we use **Latin Hypercube Sampling (LHS)** — a space-filling method that ensures points are spread evenly across the full range of each parameter. This gives us the most information about how Pythia8 behaves across the parameter space with the fewest runs.

<details>
<summary>
More information
</summary>

Each design point is one row in the design file: a set of values for the 5 Pythia8 parameters (`pT0Ref`, `ecmPow`, `coreRadius`, `coreFraction`, `CRrange`). Think of it like choosing 5-dimensional coordinates to probe the parameter space.

### Two modes

- **`Get_Design_Points = True`** — Generate `num_DP` new design points using LHS. The new points are automatically checked against all existing design files to avoid duplicates, then saved to a new numbered file `Design__Rivet__<N>.dat` in `input/Design/`.
- **`Get_Design_Points = False`** — Load the most recently generated design file from `input/Design/`. Use this when re-running model jobs without generating new parameter combinations.

### Key variables
| Variable | Description |
|----------|-------------|
| `seed` | Random seed for LHS — changing it produces a different set of design points |
| `num_DP` | Number of new design points to generate in this batch |

#

> **Background:** See **slides 26–27** for a visual explanation of Latin Hypercube Sampling and the DETMAX algorithm.


</details>


In [ ]:
# Run in apptainer kernel only

if Get_Design_Points:
    print("Generating design points via LHS.")
    os.makedirs(f"{input_dir}/Design", exist_ok=True)

    index_files  = glob.glob(f"{input_dir}/Design/Design__Rivet__*.dat")
    index_numbers = [int(f.split('__')[-1].split('.')[0]) for f in index_files]
    max_index = (max(index_numbers) if index_numbers else 0) + 1

    Design_file = f'Design__Rivet__{max_index}.dat'
    output_file = f'{input_dir}/Design/{Design_file}'
    shutil.copy(f"{input_dir}/Rivet/parameter_prior_list.dat", output_file)

    RawDesign = Reader.ReadDesign(f'{input_dir}/Rivet/parameter_prior_list.dat')
    priors, parameter_names, dim = DesignPoints.get_prior(RawDesign)

    # Collect all existing rows to avoid duplicates
    existing_rows = set()
    for oldfile in glob.glob(f"{input_dir}/Design/*.dat"):
        with open(oldfile) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                existing_rows.add(line)

    run_duplicate_check = True
    while run_duplicate_check:
        design_points = DesignPoints.get_design(num_DP, priors, seed)
        design_points = np.atleast_2d(design_points)
        current_rows  = {' '.join(f"{v:.18e}" for v in row) for row in design_points}
        if current_rows.isdisjoint(existing_rows):
            print("No duplicates detected.")
            run_duplicate_check = False
        else:
            print("Duplicates detected — re-generating.")
            seed = random.randint(1, 2**32 - 1)

    with open(output_file, 'a') as f:
        f.write(f"\n\n# LHS Seed = {seed}; Number of Design Points = {num_DP}")
        f.write('\n# Design point indices (row index): ' +
                ' '.join(str(i) for i in range(len(design_points))) + '\n')
        for row in design_points:
            f.write(' '.join(f"{v:.18e}" for v in row) + '\n')

    print(f"Wrote {len(design_points)} design points → {output_file}")

else:
    print("Loading design points from input/Design/.")

    index_files   = glob.glob(f"{input_dir}/Design/Design__Rivet__*.dat")
    index_numbers = [int(f.split('__')[-1].split('.')[0]) for f in index_files]

    if not index_numbers:
        raise FileNotFoundError("No Design files found. Set Get_Design_Points=True to generate them.")

    max_index   = max(index_numbers)
    Design_file = f'Design__Rivet__{max_index}.dat'
    RawDesign   = Reader.ReadDesign(f'{input_dir}/Design/{Design_file}')
    priors, parameter_names, dim = DesignPoints.get_prior(RawDesign)
    design_points = np.atleast_2d(RawDesign['Design'])

print(f"Design file : {Design_file}")
print(f"Parameters  : {parameter_names}")
print(f"Shape       : {design_points.shape}")

## 6. Run Pythia8 + Rivet

Runs Pythia8 and Rivet for every design point and every collision system in `Coll_System`.

<details>
<summary>
More information
</summary>

For each design point, Pythia8 simulates `nevents` collisions using the parameter values for that point. Rivet processes the simulated events through the compiled analyses and saves the measured observables to a `.yoda` file. A **YODA file** is a standard HEP data format (similar to a ROOT file but simpler) that stores histograms from the analysis.

### Execution Modes

| Flag | Behavior |
|------|----------|
| `Submit_Job_Model` | Submits SLURM batch jobs to run Pythia8 + Rivet across all design points. Recommended for full production runs. |
| `Rerun_job_model` | Resubmits SLURM jobs for specific design points listed in `DP_LIST`. Use to recover failed or incomplete runs without rerunning everything. |
| `Run_Model` | Runs Pythia8 + Rivet interactively for each design point. Use for small tests only — not suitable for large design point sets. |

### Key parameters

#

| Parameter | Description |
|-----------|-------------|
| `nevents` | Number of simulated collision events per design point. More events = less statistical noise in the output, but longer runtime. 1,000 is fast for testing; 100,000+ for production. |
| `model_seed` | Random seed for the Pythia8 event generator. Fixing it makes runs reproducible. |
| `PT_Min / PT_Max` | Optional transverse momentum cuts on generated events. `-1` means no cut (use all events). |

#

### Runtime

This is the most time-consuming step. For small tests (`num_DP = 5`, `nevents = 1000`) it takes a few minutes per design point. For production runs (100+ design points, 100k+ events), use `Submit_Job_Model` instead of running interactively here.

> **Background:** See **slides 25 and 28–30** for an overview of the Monte Carlo method, Pythia8's physics model (MPI, hadronization, color reconnection), and alternative generators JETSCAPE and Herwig.


</details>

In [ ]:
# Run for both kernels

model_seed  = 283    # model seed

if RUN_PT_HAT_BINS:
    PT_EDGES    = [0, 15, 20, 25, 30, 40, 60]
    pt_hat_flag = "true" 
    num_tasks = len(PT_EDGES)
else:
    PT_Min      = -1
    PT_Max      = -1
    pt_hat_flag = "false"
    num_tasks = 7

if Submit_Job_Model:
    if get_kernel() == 'host':
        for system in Coll_System:
            if system not in analyses_list:
                print(f"No analyses defined for system: {system} — skipping.")
                continue

            pt_edges_arg = "-1 -1" if not RUN_PT_HAT_BINS else ' '.join(str(e) for e in PT_EDGES)
            max_idx      = min(num_DP - 1, 11)

            !sbatch --parsable --array=0-{max_idx} --ntasks={num_tasks} --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                    {hpc_dir}/run_rivet.slurm {input_dir} {pt_hat_flag} "{pt_edges_arg}" {project} {system} {num_DP} {nevents} {CONTAINER} {BIND_PATH}
    else:
        print("⚠️ Switch to Isaac (Host) kernel to submit jobs.")

elif Rerun_job_model:
    if get_kernel() == 'host':
        for system in Coll_System:
            if system not in analyses_list:
                print(f"No analyses defined for system: {system} — skipping.")
                continue

            DP_LIST     =[54, 113]
            pt_edges_arg = "-1 -1" if not RUN_PT_HAT_BINS else ' '.join(str(e) for e in PT_EDGES)
            dp_list_arg  = ' '.join(str(d) for d in DP_LIST)
            array_spec   = f"0-{len(DP_LIST) - 1}"

            !sbatch --parsable --array={array_spec} --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                    {hpc_dir}/run_rivet_rerunDP.slurm {input_dir} {pt_hat_flag} "{pt_edges_arg}" "{dp_list_arg}" {project} {system} {num_DP} {nevents} {CONTAINER} {BIND_PATH}
    else:
        print("⚠️ Switch to Isaac (Host) kernel to submit jobs.")
        
elif Run_Model:
    for system in Coll_System:
        if system not in analyses_list:
            print(f"No analyses defined for system: {system} — skipping.")
            continue

        System, Energy = system.split('_')
        system_analyses = analyses_list[system]
        if not system_analyses:
            print(f"Empty analyses list for {system} — skipping.")
            continue

        print(f"Running {model} for system: {system}")
        for i, point in enumerate(design_points):
            print(f"  DP {i+1}: {point}")
            param_tag = DesignPoints.generate_param_tag(parameter_names, point)
            merge_tag = f"DP_{i+1}"

            subprocess.run([
                'bash', f'/usr/local/share/Bayes_HEP/Design_Points/Models/{model}/run_{model}.sh',
                ','.join(system_analyses), Rivet_dir, project_dir, System, Energy, str(nevents), str(model_seed), param_tag, merge_tag, str(PT_Min), str(PT_Max)], check=True)


## 7. Merge .yoda Files and Generate HTML Reports

After the Pythia8 + Rivet runs, each design point has one or more `.yoda` files — one per run of the event generator. This step **merges** those files (combining statistics from all runs) and builds an **HTML report** for each design point.

Merged outputs and HTML reports are saved under `rivet/Models/pythia8/html_reports/`.

<details>
<summary>
More Information
</summary>

The HTML report is a browsable web page showing plots of all measured observables for that design point, overlaid with the experimental reference data. You can open it in a browser to visually check how well (or poorly) that parameter set matches the data — this is a useful sanity check before investing time in the full emulator training.

</details>

In [ ]:
# Run in apptainer kernel only

if Submit_Merge_job:
    if get_kernel() == 'host':
        for system in Coll_System:
            if system not in analyses_list:
                print(f"No analyses defined for system: {system} — skipping.")
                continue

            pt_edges_arg = "-1 -1" if not RUN_PT_HAT_BINS else ' '.join(str(e) for e in PT_EDGES)
            max_idx      = min(num_DP - 1, 11)
            
            !sbatch --parsable --array=0-0 --ntasks=1 --qos={QOS} --partition={QOS} --time={WALLTIME} --error={error_path} --output={output_path} \
                {hpc_dir}/run_rivet_merge.slurm {input_dir} {pt_hat_flag} "{pt_edges_arg}" {project} {system} {num_DP} {nevents} {CONTAINER} {BIND_PATH}

elif Rivet_Merge:
    for system in Coll_System:
        System, Energy = system.split('_')
        system_analyses = analyses_list[system]

        if not system_analyses:
            print(f"No analyses for {system} — skipping.")
            continue

        print(f"Merging + HTML for system: {system}")
        for i, point in enumerate(design_points):
            merge_tag = f"DP_{i+1}"
            print(f"  {merge_tag}")
            
            # Merge results
            subprocess.run(['bash', '/usr/local/share/Bayes_HEP/Design_Points/Rivet_Analyses/merge.sh', project_dir, model, System, Energy, merge_tag, str(equiv_on)], check=True)
        
            # Generate HTML report
            subprocess.run(['bash', '/usr/local/share/Bayes_HEP/Design_Points/Rivet_Analyses/mkhtml.sh', project_dir, model, System, Energy, merge_tag], check=True)

    print("Merge + HTML generation complete.")

## 8. Write Data and Prediction Input Files

Parses the HTML reports and writes the numerical observable values to plain-text files that `Bayes_Notebook.ipynb` reads.

<details>
<summary>
More information
</summary>

Two sets of files are written:
- **`input/Data/`** — experimental measurements from the reference data embedded in each Rivet analysis (x-values, y-values, and uncertainties from the real detector)
- **`input/Prediction/`** — Pythia8-simulated values at each design point for each observable

These files are the bridge between this notebook and `Bayes_Notebook.ipynb`. The emulator in the Bayes notebook reads `Prediction/` files to train on, and compares against `Data/` files during calibration.

**After this cell completes successfully, you are ready to open `Bayes_Notebook.ipynb`.**

</details>

In [ ]:
# Run in apptainer kernel only

if Write_input_Rivet:
    os.makedirs(f"{input_dir}/Data",       exist_ok=True)
    os.makedirs(f"{input_dir}/Prediction", exist_ok=True)

    for system in Coll_System:
        System, Energy = system.split('_')
        system_analyses = analyses_list[system]
        print(f"Writing Data/Prediction files for: {system}")

        for i, point in enumerate(design_points):
            DP = i + 1
            for analysis in system_analyses:
                for hist in tagged_analyses[system][analysis]:
                    base = (f"{project_dir}/Models/{model}/html_reports/{model}_{System}_{Energy}_DP_{DP}_report.html/{analysis}/{hist}")
                    datafile  = base + "__data.py"
                    labelfile = base + ".py"

                    obs, subobs, x_scale, y_scale, x_lims, y_lims, title = RivetParser.extract_labels(labelfile)

                    input_data_name = (f"{input_dir}/Data/Data__{Energy}__{System}__{analysis}__{hist}")
                    input_pred_name = (f"{input_dir}/Prediction/Prediction__{model}__{Energy}__{System}__{analysis}__{hist}__DG_{max_index}")

                    RivetParser.extract_data(datafile, model, input_data_name, input_pred_name,
                         obs, subobs, DP,
                         x_scale=x_scale, y_scale=y_scale,
                         x_lims=x_lims, y_lims=y_lims,
                         title=title)

    RivetParser._flush_all_predictions()

    print("Data/Prediction files written.")